Install BM25 Package

In [ ]:
!pip install -q rank_bm25

Import Required Libraries

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

Create BM25 Retriever

In [ ]:
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4

Create Hybrid Retriever

In [ ]:
docs = bm25_retriever.invoke(
    "What was Apple's total revenue in fiscal year 2023?"
)

for d in docs[:2]:
    print(d.page_content[:300])

Apple's gross margin percentage was 44.1% in fiscal 2023, compared to 43.3% in fiscal 2022.
Services revenue reached an all-time high of $85.2 billion in fiscal 2023, up 9 percent year over year.
PRODUCTS AND SERVICES
Apple designs, manufactures and markets smartphones, personal computers, tablets, wearables and accessories.
iPhone is Apple's line of smartphones based on its iOS operating system.
iPhone net sales were $200.6 billion in fiscal 2023, representing approximately 52% of total rev


In [ ]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[
        retriever,       # FAISS retriever
        bm25_retriever   # BM25 retriever
    ],
    weights=[0.6, 0.4]
)

Test Hybrid Retrieval

In [ ]:
query = "How much cash did Apple have in 2023?"

docs = hybrid_retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\n--- Doc {i+1} ---")
    print(doc.page_content[:400])


--- Doc 1 ---
iPad net sales were $28.3 billion in fiscal 2023.
Wearables, Home and Accessories net sales were $39.8 billion in fiscal 2023.
Apple's Services segment includes advertising, AppleCare, cloud, digital content, payment and other services.
The App Store, Apple Music, Apple TV+, Apple Arcade, iCloud and Apple Pay are key Services offerings.
The Company had approximately 2.2 billion active devices at t

--- Doc 2 ---
LIQUIDITY AND CAPITAL RESOURCES
The Company believes its existing balances of cash, cash equivalents and unrestricted marketable securities,
together with cash generated by operations, will be sufficient to satisfy its expected cash needs.
Cash and cash equivalents as of September 30, 2023 were $29.965 billion.
Total marketable securities were $100.544 billion, consisting of current marketable sec

--- Doc 3 ---
resources, as well as from new market entrants.
Apple depends on the performance of distributors, carriers, wholesalers and other resellers.
The Company'

In [ ]:
# Comparision
retriever.invoke(query)

[Document(metadata={'source': 'Apple_10K_2023_Risk'}, page_content="Apple's gross margin percentage was 44.1% in fiscal 2023, compared to 43.3% in fiscal 2022.\nServices revenue reached an all-time high of $85.2 billion in fiscal 2023, up 9 percent year over year."),
 Document(metadata={'source': 'Apple_10K_2023_Risk'}, page_content="resources, as well as from new market entrants.\nApple depends on the performance of distributors, carriers, wholesalers and other resellers.\nThe Company's fiscal year 2023 revenue was $383.3 billion, compared to $394.3 billion in fiscal 2022,\na decrease of approximately 2.8 percent.\nThe Company's net income for fiscal 2023 was $97.0 billion, or $6.13 diluted earnings per share,\ncompared to $99.8 billion, or $6.11 diluted earnings per share, in fiscal 2022."),
 Document(metadata={'source': 'Apple_10K_2023_Products'}, page_content="iPad net sales were $28.3 billion in fiscal 2023.\nWearables, Home and Accessories net sales were $39.8 billion in fiscal 2

Create Hybrid RAG Chain

In [ ]:
hybrid_rag_chain = (
    {
        "context": hybrid_retriever,
        "question": RunnablePassthrough()
    }
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

Benchmark Hybrid RAG

In [ ]:
for query in TEST_QUERIES:

    start = time.time()

    answer = hybrid_rag_chain.invoke(query)

    latency = time.time() - start

    print(f"\nQuestion: {query}")
    print(f"Latency: {latency:.2f}s")
    print(f"Answer: {answer[:300]}")


Question: What was Apple's total revenue in fiscal year 2023?
Latency: 1.90s
Answer: Apple's total revenue in fiscal year 2023 was $383.3 billion. 

(Source: Apple_10K_2023_Risk)

Question: How much cash did Apple have at the end of fiscal 2023?
Latency: 1.12s
Answer: Apple had $29.965 billion in cash and cash equivalents at the end of fiscal 2023. 

(Source: Apple_10K_2023_Liquidity)

Question: What percentage of Apple's revenue came from iPhone in 2023?
Latency: 2.24s
Answer: iPhone net sales were $200.6 billion in fiscal 2023, representing approximately 52% of Apple's total revenue. 

Source: Apple_10K_2023_Products

Question: How much did Apple return to shareholders in fiscal 2023?
Latency: 0.94s
Answer: Apple returned over $77 billion to shareholders in fiscal 2023, including $15.1 billion in dividends and dividend equivalents and $62.2 billion through repurchases of 471 million shares. 

(Source: Apple_10K_2023_Liquidity)

Question: What is Apple's gross margin for fiscal 2023?

Evaluate with RAGAS

In [ ]:
contexts_used = []

for q in TEST_QUERIES:
    docs = hybrid_retriever.invoke(q)
    contexts_used.append([d.page_content for d in docs])

Re-Ranking

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import CrossEncoder

In [ ]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Re-rank Retrieved Documents

In [ ]:
def rerank(query, docs, top_k=4):

    pairs = [
        [query, d.page_content]
        for d in docs
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(scores, docs),
        reverse=True,
        key=lambda x: x[0]
    )

    return [doc for score, doc in ranked[:top_k]]

Test Re-ranking

In [ ]:
query = "What percentage of revenue came from iPhone?"

docs = hybrid_retriever.invoke(query)

reranked_docs = rerank(query, docs)

for d in reranked_docs:
    print(d.page_content[:400])

PRODUCTS AND SERVICES
Apple designs, manufactures and markets smartphones, personal computers, tablets, wearables and accessories.
iPhone is Apple's line of smartphones based on its iOS operating system.
iPhone net sales were $200.6 billion in fiscal 2023, representing approximately 52% of total revenue.
Mac net sales were $29.4 billion in fiscal 2023, down from $40.2 billion in fiscal 2022.
iPad 
Apple's gross margin percentage was 44.1% in fiscal 2023, compared to 43.3% in fiscal 2022.
Services revenue reached an all-time high of $85.2 billion in fiscal 2023, up 9 percent year over year.
resources, as well as from new market entrants.
Apple depends on the performance of distributors, carriers, wholesalers and other resellers.
The Company's fiscal year 2023 revenue was $383.3 billion, compared to $394.3 billion in fiscal 2022,
a decrease of approximately 2.8 percent.
The Company's net income for fiscal 2023 was $97.0 billion, or $6.13 diluted earnings per share,
compared to $99.8 b
iP

In [ ]:
print('Extension task: complete the hybrid retrieval implementation above!')

Extension task: complete the hybrid retrieval implementation above!
